# 3. Prediksi Harga 6 Bulan
Memprediksi harga penutupan bulanan tiap komoditas (rata-rata semua pasar). Empat model dibandingkan lewat uji mundur 24 bulan terakhir, lalu model dengan kesalahan terkecil dipakai untuk prediksi 6 bulan ke depan beserta rentang ketidakpastiannya.

In [ ]:
import re

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, f1_score, silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.tree import DecisionTreeClassifier

import os
import matplotlib.pyplot as plt

data_path = '../data/processed/data_bersih.csv'
output_dir = '../data/processed'
item_grafik = 'rice'
bulan_depan = 6
uji_bulan = 24

In [ ]:
KOL_PASAR = 'mkt_name'
KOL_TANGGAL = 'date'
KOL_INDEKS = 'food_price_index'
KOMODITAS = ['eggs', 'garlic', 'meat_beef', 'meat_chicken', 'meat_chicken_broiler', 'oil', 'onions', 'rice', 'sugar']

BATAS_WASPADA = 5
BATAS_LONJAKAN = 10
NAMA_KELAS = ['Normal', 'Waspada', 'Lonjakan']
FITUR_DASAR = ['inflasi_ini', 'inflasi_lalu', 'bulan']
PERSEN_TEST = 0.2
TAHUN_ACUAN_AKHIR = 2019

BOBOT_RISIKO = 0.6
BOBOT_MOMENTUM = 0.2
BOBOT_VOLATILITAS = 0.2
BATAS_PRIORITAS = 50
BATAS_PANTAU = 25
BATAS_HARGA_PERSEN = 5

NAMA_KOMODITAS = {
    'eggs': 'Telur', 'garlic': 'Bawang Putih', 'meat_beef': 'Daging Sapi', 'meat_chicken': 'Daging Ayam',
    'meat_chicken_broiler': 'Daging Ayam Broiler', 'oil': 'Minyak Goreng', 'onions': 'Bawang Merah',
    'rice': 'Beras', 'sugar': 'Gula', 'food_price_index': 'Indeks Pangan',
}
MUSIM_ONI = ['DJF', 'JFM', 'FMA', 'MAM', 'AMJ', 'MJJ', 'JJA', 'JAS', 'ASO', 'SON', 'OND', 'NDJ']

In [ ]:
def model_datar(y, h):
    return np.repeat(y[-1], h)


def model_teredam(y, h, k=6, phi=0.8):
    kemiringan = np.mean(np.diff(y[-(k + 1):]))
    return y[-1] + kemiringan * np.cumsum(phi ** np.arange(1, h + 1))


def model_regresi(y, h, derajat, jendela=36):
    y = y[-jendela:]
    X = np.arange(len(y)).reshape(-1, 1)
    model = make_pipeline(PolynomialFeatures(degree=derajat), LinearRegression()).fit(X, y)
    return model.predict(np.arange(len(y), len(y) + h).reshape(-1, 1))


def kandidat_prediksi():
    return {
        'Harga tetap': model_datar,
        'Tren teredam': model_teredam,
        'Linear (36 bulan)': lambda y, h: model_regresi(y, h, 1),
        'Polinomial derajat 2 (36 bulan)': lambda y, h: model_regresi(y, h, 2),
    }


def prediksi_harga(seri, bulan_depan=6, uji_bulan=24):
    s = seri.dropna().sort_index().asfreq('MS').interpolate(limit_direction='both')
    y = s.values.astype(float)
    if len(y) < uji_bulan + bulan_depan + 12:
        return None
    kandidat = kandidat_prediksi()
    galat = {nama: [] for nama in kandidat}
    for o in range(len(y) - uji_bulan - bulan_depan + 1, len(y) - bulan_depan + 1):
        aktual = y[o:o + bulan_depan]
        for nama, f in kandidat.items():
            galat[nama].append(np.abs((aktual - f(y[:o], bulan_depan)) / aktual))
    galat = {nama: np.array(v) for nama, v in galat.items()}
    tabel = pd.DataFrame({'MAPE (%)': {nama: v.mean() * 100 for nama, v in galat.items()}})
    terbaik = tabel['MAPE (%)'].idxmin()
    lebar = np.maximum.accumulate(np.quantile(galat[terbaik], 0.9, axis=0))
    tanggal = pd.date_range(s.index[-1] + pd.offsets.MonthBegin(1), periods=bulan_depan, freq='MS')
    pred = kandidat[terbaik](y, bulan_depan)
    return {'model': terbaik, 'mape': float(tabel.loc[terbaik, 'MAPE (%)']), 'tabel': tabel.round(2),
            'prediksi': pd.Series(pred, index=tanggal), 'bawah': pd.Series(pred * (1 - lebar), index=tanggal),
            'atas': pd.Series(pred * (1 + lebar), index=tanggal)}

## Data bulanan
Setiap bulan dirata-ratakan dari semua pasar. Kolom c_ adalah harga penutupan bulan itu.

In [ ]:
df = pd.read_csv(data_path, parse_dates=[KOL_TANGGAL])
df['bulan'] = df[KOL_TANGGAL].dt.to_period('M').dt.to_timestamp()
print(f'Data: {df["bulan"].min():%b %Y} sampai {df["bulan"].max():%b %Y}')

## Uji mundur semua komoditas
Untuk tiap bulan dari 24 bulan terakhir, model dilatih memakai data sebelum bulan itu lalu diminta menebak 6 bulan berikutnya. MAPE adalah rata-rata kesalahan persentase.

In [ ]:
hasil_semua = {}
baris = []
for item in KOMODITAS:
    seri = df.groupby('bulan')[f'c_{item}'].mean()
    hasil = prediksi_harga(seri, bulan_depan, uji_bulan)
    hasil_semua[item] = (seri, hasil)
    baris.append({'Komoditas': item, 'Model terpilih': hasil['model'], 'MAPE terpilih (%)': hasil['mape']}
                 | {nama: v for nama, v in hasil['tabel']['MAPE (%)'].items()})
ringkas = pd.DataFrame(baris).set_index('Komoditas')
ringkas.round(2)

Model sederhana sering sulit dikalahkan pada harga pangan. Bila tebakan `Harga tetap` menang di suatu komoditas, itu hasil yang apa adanya dan ditampilkan seperti itu di dashboard.

In [ ]:
ringkas['Model terpilih'].value_counts()

## Candlestick dan prediksi
Satu batang per bulan: badan batang dari harga awal ke harga akhir, garis tipis dari harga terendah ke tertinggi. Hijau berarti harga akhir lebih tinggi dari harga awal.

In [ ]:
seri, hasil = hasil_semua[item_grafik]
ohlc = df.groupby('bulan')[[f'{p}_{item_grafik}' for p in 'ohlc']].mean().tail(48)
ohlc.columns = list('ohlc')
fig, ax = plt.subplots(figsize=(13, 6))
for tgl, r in ohlc.iterrows():
    warna = '#2e7d32' if r['c'] >= r['o'] else '#c62828'
    ax.vlines(tgl, r['l'], r['h'], color=warna, linewidth=1)
    ax.bar(tgl, abs(r['c'] - r['o']), bottom=min(r['o'], r['c']), width=20, color=warna)
ax.plot(hasil['prediksi'].index, hasil['prediksi'].values, color='#1565c0', linestyle='dashed', marker='o', markersize=3, label=f'Prediksi ({hasil["model"]})')
ax.fill_between(hasil['prediksi'].index, hasil['bawah'].values, hasil['atas'].values, color='#1565c0', alpha=0.2, label='Rentang ketidakpastian')
ax.set_title(f'Harga {NAMA_KOMODITAS[item_grafik]}: candlestick bulanan dan prediksi {bulan_depan} bulan')
ax.set_xlabel('Bulan')
ax.set_ylabel('Harga (IDR/kg)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## Simpan hasil

In [ ]:
ringkas.to_csv(f'{output_dir}/metrik_prediksi_harga.csv')
forecast = pd.concat({item: pd.DataFrame({'prediksi': h['prediksi'], 'bawah': h['bawah'], 'atas': h['atas']}) for item, (s, h) in hasil_semua.items()},
                     names=['komoditas', 'bulan']).round(0)
forecast.to_csv(f'{output_dir}/hasil_prediksi_harga.csv')
print('Metrik dan prediksi tersimpan di', output_dir)